In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch
from velovi import preprocess_data, VELOVI
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd
import unitvelo as utv #pip install unitvelo
import time
from os.path import exists
method = 'veloVI'

(Running UniTVelo 0.2.5.2)
2024-12-17 03:11:35


2024-12-16 22:11:36.247045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-16 22:11:36.271923: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-16 22:11:36.279457: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
#
def add_velovi_outputs_to_adata(adata, vae):
    latent_time = vae.get_latent_time(n_samples=25)
    velocities = vae.get_velocity(n_samples=25, velo_statistic="mean")

    t = latent_time
    scaling = 20 / t.max(0)

    adata.layers["velocity"] = velocities / scaling
    adata.layers["latent_time_velovi"] = latent_time

    adata.var["fit_alpha"] = vae.get_rates()["alpha"] / scaling
    adata.var["fit_beta"] = vae.get_rates()["beta"] / scaling
    adata.var["fit_gamma"] = vae.get_rates()["gamma"] / scaling
    adata.var["fit_t_"] = (
        torch.nn.functional.softplus(vae.module.switch_time_unconstr)
        .detach()
        .cpu()
        .numpy()
    ) * scaling
    adata.layers["fit_t"] = latent_time.values * scaling[np.newaxis, :]
    adata.var['fit_scaling'] = 1.0

In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

In [ ]:
for dataset in datasets:
    print(dataset)
    adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    #setup data and train model
    start = time.time()
    scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    adata = preprocess_data(adata)
    VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
    vae = VELOVI(adata)
    vae.train()
    add_velovi_outputs_to_adata(adata, vae)
    end = time.time()
    scv.pp.neighbors(adata)
    scv.tl.velocity_graph(adata)
    scv.tl.velocity_embedding(adata)
    fix, ax = plt.subplots(1, 1, figsize = (8, 6))
    scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity',color="clusters",
                                     show = False, ax = ax)
    plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
    # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    metrics = utv.evaluate(adata, ground_truth, 'clusters', 'velocity')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDir_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB=df_CB.append(pd.DataFrame([[np.mean(cb_score), end-start]],columns=df_CB.columns,index=[dataset]))
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    df_IC=df_IC.append(pd.DataFrame([[np.mean(IC_score), end-start]],columns=df_IC.columns,index=[dataset]))
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [8]:
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')